# Temporal Context: does k = 7 close any of the gap?

Re-trains `pcrtc/09` with **all seven** Sentinel-1 acquisitions per patch
instead of three. Single-variable change: `CONTEXT_K` only.

**Why this is worth a run.** Tessa's best Sentinel-2 checkpoint is
`linear_k6_att_best.pth` -- **k = 6**. Every comparison in this study
against that ~0.74-0.78 reference has been made with a Sentinel-1 model
using **k = 3**, so the S1 model has had roughly half the temporal
context of the baseline it is being measured against. That is a confound
sitting underneath the central question of the dissertation, and it is
cheap to remove.

**The data support it fully.** All 1676 paired patches have exactly 7
views on disk, so raising `CONTEXT_K` costs no patches -- this is an
upgrade, not a trade.

**What changes, and what does not.**

| | value | note |
|---|---|---|
| `CONTEXT_K` | 3 -> **7** | the variable under test |
| `cond_channels` | 12 -> 28 | `4 * CONTEXT_K`, already parameterised |
| `attr_dim` | 24 -> 56 | `8 * CONTEXT_K` |
| `SPLIT_SEED` | 42, fixed | identical validation patches to `09` |
| `INIT_SEED` | 42 | matches `09`'s own seed, so `k` is the only deliberate difference |
| everything else | identical to `09` | data, epochs, LR, schedule, sampler, metrics |

**Interpret against the variance floor, not against zero.** Two runs
differing only in initialisation land ~0.05 ZNCC apart. A k = 7 result
inside that band is not evidence of anything; run `dem_unet/06` first so the
threshold is measured rather than assumed.

**A plausible mechanism either way.** More looks at the same ground should
average down speckle, which is the dominant noise in a single SAR
acquisition -- that argues for a gain. Against it: the seven acquisitions
span a window over which surface conditions change, so the extra views may
add variance rather than signal. Both are worth stating in the write-up
whichever way the number falls.

**This notebook does not execute automatically. Run cells top to bottom.**

## GPU configuration

In [ ]:
import os
import sys
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Configuration

Every value below matches `pcrtc/09` except `INIT_SEED`. Change only
`INIT_SEED` if you want to run a third replicate later.

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 42     # MUST stay 42 -- identical split to 09
INIT_SEED = 42      # matches 09; CONTEXT_K is the variable here

CONTEXT_K = 7      # <-- the variable under test (09 used 3)
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0

CHECKPOINT_NAME = f's1_{REGION}_pcrtc_realattrs_spatialsplit_k{CONTEXT_K}_unet_best.pth'
METRICS_FILENAME = f's1_pcrtc_realattrs_spatialsplit_k{CONTEXT_K}_validation_metrics.json'
BASELINE_METRICS = 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'      # 09
DEM_METRICS = 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json'       # 03 (inert DEM)

print(f'pcrtc/09 with CONTEXT_K={CONTEXT_K} (09 used 3), SPLIT_SEED={SPLIT_SEED}')
n_views = sorted({len(list(p.glob('t*.tif'))) for p in S1_DIR.glob('s1_patch_*') if p.is_dir()})
print(f'Views available per patch across the dataset: {n_views}')
assert min(n_views) >= CONTEXT_K, f'Some patches have fewer than {CONTEXT_K} views.'
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)

## Imports and seeding

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(INIT_SEED)   # NOT SPLIT_SEED
print('Global seed set to', INIT_SEED)

## Dataset adapter -- unchanged from `09`

In [ ]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_norm = (acq_date - LIDAR_SURVEY_DATE).days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256)):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{patch_id}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
                's1': condition.float(), 'attrs': attrs,
                'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Spatial-block split -- `SPLIT_SEED`, not `INIT_SEED`

Copied verbatim from `09`. The only edit is `random.Random(SPLIT_SEED)`
in place of `random.Random(SEED)`, which keeps the split pinned to 42
while the initialisation seed varies. The assertion below is the guard
that makes this whole experiment valid.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

block_ids = list(blocks.keys())
random.Random(SPLIT_SEED).shuffle(block_ids)   # pinned to 42, independent of INIT_SEED

target_val_patches = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Split: train={len(train_ids)}, val={len(val_ids)}, dropped={len(dropped_buffer)}')
assert (len(train_ids), len(val_ids)) == (534, 255), (
    f'Split is {len(train_ids)}/{len(val_ids)}, expected 534/255. The split has drifted '
    f'from 09 and no comparison below would be valid. Stop and investigate.')
print('Split confirmed identical to 09.')

train_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW)
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Model, scheduler, optimizer

In [ ]:
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K,
                        base_channels=128, embed_dim=256, unet_depth=4,
                        attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Training loop -- identical to `09`

`09`'s best validation loss arrived at **epoch 93 of 100**, with validation
wandering in the 0.0137-0.0174 band throughout and rising streaks of up to
3 epochs. Early validation movement carries no signal here. Do not stop
this run early.

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
best_epoch = -1
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            noisy = scheduler.q_sample(target, timestep)
            prediction = model(noisy, condition, attrs, timestep)
            loss = masked_mse(prediction, target, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            with autocast():
                prediction = model(scheduler.q_sample(target, timestep), condition, attrs, timestep)
                val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch + 1
        torch.save({'model_state_dict': model.state_dict(),
                    'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS,
                               'noise_schedule': NOISE_SCHEDULE, 'region': REGION,
                               'split_seed': SPLIT_SEED, 'init_seed': INIT_SEED},
                    'epoch': best_epoch, 'val_loss': val_loss},
                   CHECKPOINT_DIR / CHECKPOINT_NAME)

print(f'\nBest val {best_val:.6f} at epoch {best_epoch}  (09 reached 0.013706 at epoch 93)')

## Evaluation -- identical metric suite and sampler

In [ ]:
checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            metric_rows.append({
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            })

metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as h:
    json.dump(metric_rows, h, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {k: float(np.nanmean([r[k] for r in metric_rows])) for k in metric_rows[0] if k != 'patch_id'})

## Comparison against `09` (k = 3)

Identical validation patches, identical everything except `CONTEXT_K`.
The bootstrap CI covers **patch sampling**, not training variance -- so the
last block checks the difference against the measured run-to-run floor
from `dem_unet/06`. A k = 7 gain inside that floor is not a result.

In [ ]:
def load_rows(name):
    p = OUTPUT_DIR / name
    if not p.exists():
        print(f'  (missing: {name})')
        return None
    return {r['patch_id']: r for r in json.load(p.open())}

runs = {'09  (k=3)': load_rows(BASELINE_METRICS),
        f'k={CONTEXT_K}': {r['patch_id']: r for r in metric_rows}}
seed_rep = load_rows('s1_pcrtc_realattrs_spatialsplit_seed43_validation_metrics.json')
if seed_rep:
    runs['dem_unet/06 seed rep (k=3)'] = seed_rep
runs = {k: v for k, v in runs.items() if v}

common = set.intersection(*(set(v) for v in runs.values()))
print(f'runs: {len(runs)}   patches common to all: {len(common)}\n')

METRICS = ['zncc', 'rmse_m', 'psd_rmse', 'jsd', 'sigma_error_pct', 'gt_std_val', 'pred_std_val']
print(f"{'metric':<18}" + ''.join(f'{k:>24}' for k in runs))
for m in METRICS:
    print(f'{m:<18}' + ''.join(f'{np.nanmean([r[i][m] for i in common]):>24.4f}' for r in runs.values()))

rng = np.random.default_rng(42)
base = runs['09  (k=3)']
this = runs[f'k={CONTEXT_K}']
print('\n--- k=7 vs k=3, paired over the same patches ---')
for m in ['zncc', 'rmse_m', 'psd_rmse', 'jsd']:
    d = np.array([this[i][m] - base[i][m] for i in common])
    d = d[np.isfinite(d)]
    boot = np.array([rng.choice(d, d.size, replace=True).mean() for _ in range(10000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f'{m:<12} mean diff {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  '
          f'k=7 higher on {float((d > 0).mean()):.0%}')

if seed_rep:
    dv = np.array([seed_rep[i]['zncc'] - base[i]['zncc'] for i in common])
    floor = abs(float(np.nanmean(dv)))
    dz = float(np.nanmean([this[i]['zncc'] - base[i]['zncc'] for i in common]))
    print(f'\nMeasured run-to-run |delta ZNCC| floor: {floor:.4f}')
    print(f'k=7 delta ZNCC: {dz:+.4f}  ->  '
          f'{"EXCEEDS the noise floor -- a real effect" if abs(dz) > floor else "INSIDE the noise floor -- not distinguishable from a reseed"}')
else:
    print('\nRun dem_unet/06 (seed replicate) to get the noise floor before reading the CI above')
    print('as evidence: the bootstrap covers patch sampling, not training variance.')